1、获取top300项目名称信息

In [ ]:
import requests
import csv

url = "https://oss.x-lab.info/open_leaderboard/open_rank/repo/global/2025.json"

resp = requests.get(url)
data = resp.json()

records = data.get("data", [])

rows = []
for entry in records:
    item = entry.get("item", {})
    repo_name = item.get("name", "")
    company = item.get("company", "")  # 若无则为空
    rank = entry.get("rank", "")
    value = entry.get("value", "")
    rankDelta = entry.get("rankDelta", "")
    valueDelta = entry.get("valueDelta", "")

    rows.append([
        repo_name, company, rank, value, rankDelta, valueDelta
    ])

# 保存 CSV
csv_file = "top300_projects.csv"
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["repo_name", "company", "rank", "value", "rankDelta", "valueDelta"])
    writer.writerows(rows)

print(f"已成功保存到 {csv_file}，共 {len(rows)} 条记录")


2、获取每个项目的各个指标

In [16]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# -------- 1. 读取 top300 项目列表 --------
xlsx_path = "top300_projects.xlsx"
df = pd.read_excel(xlsx_path)

# 指标列表（你提供的全部）
metrics_list = [
    "active_dates_and_times",
    "activity_details",
    "activity",
    "attention",
    "bus_factor_detail",
    "bus_factor",
    "change_request_age",
    "change_request_resolution_duration",
    "change_requests_accepted",
    "change_request_response_time",
    "change_requests_reviews",
    "change_requests",
    "code_change_lines_add",
    "code_change_lines_remove",
    "code_change_lines_sum",
    "contributor_email_suffixes",
    "inactive_contributors",
    "issue_age",
    "issue_comments",
    "issue_resolution_duration",
    "issue_response_time",
    "issues_and_change_request_active",
    "issues_closed",
    "issues_new",
    "new_contributors_detail",
    "new_contributors",
    "openrank",
    "participants",
    "stars",
    "technical_fork"
]

# -------- 2. 下载函数 --------
def download_metric(company, project, metric):
    url = f"https://oss.open-digger.cn/github/{company}/{project}/{metric}.json"
    save_dir = f"data/origin/{company}/{project}"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{metric}.json")

    # 已存在就不重新下载
    if os.path.exists(save_path):
        return f"[SKIP] {save_path}"

    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(resp.content)
            return f"[OK] {save_path}"
        else:
            return f"[404] {url}"
    except Exception as e:
        return f"[ERROR] {url} -> {e}"

# -------- 3. 进度条 & 多线程批量下载 --------
tasks = []

# 任务总数
total_tasks = len(df) * len(metrics_list)

# 用 tqdm 显示总任务进度
with tqdm(total=total_tasks, desc="Total Progress", unit="file") as pbar:
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = []

        for _, row in df.iterrows():
            project = str(row["repo_name"]).split("/")[-1]
            company = row["company"]

            for metric in metrics_list:
                futures.append(
                    executor.submit(download_metric, company, project, metric)
                )

        # 等待完成 + 更新进度条
        for future in as_completed(futures):
            result = future.result()
            pbar.set_postfix_str(result)  # 显示当前下载信息
            pbar.update(1)

print("\n🎉 All downloads finished with progress bar!")


Total Progress: 100%|██████████████████| 9000/9000 [02:51<00:00, 52.34file/s, [OK] data/elementor/elementor\stars.json]


🎉 All downloads finished with progress bar!


3、从company/project/metrics ————>  metrics/company_project.json

In [3]:
import os
import json
import shutil

origin_root = "data/origin"
processing_root = "data/processing"

# 创建 processing 根目录
os.makedirs(processing_root, exist_ok=True)

summary = {}   # 记录每个指标文件数量
errors = []    # 存放错误日志

for company in os.listdir(origin_root):
    company_path = os.path.join(origin_root, company)
    if not os.path.isdir(company_path):
        continue

    for project in os.listdir(company_path):
        project_path = os.path.join(company_path, project)
        if not os.path.isdir(project_path):
            continue

        for metric_file in os.listdir(project_path):
            if not metric_file.endswith(".json"):
                continue

            metric_name = metric_file.replace(".json", "")
            src_file = os.path.join(project_path, metric_file)

            # 输出目录：data/processing/activity/
            metric_output_dir = os.path.join(processing_root, metric_name)
            os.makedirs(metric_output_dir, exist_ok=True)

            # 输出文件：company_project.json
            target_filename = f"{company}_{project}.json"
            dst_file = os.path.join(metric_output_dir, target_filename)

            try:
                shutil.copy2(src_file, dst_file)

                # 统计成功次数
                summary[metric_name] = summary.get(metric_name, 0) + 1

            except Exception as e:
                errors.append(f"{src_file} -> {e}")

print("\n🎉 数据整理完成！")
print("====== 指标文件汇总 ======")

for metric, count in sorted(summary.items()):
    print(f"{metric}: {count} files")

print("\n====== 错误文件 ======")
if errors:
    for err in errors[:10]:  # 只展示前 10 条
        print("❌", err)
    print(f"... 共 {len(errors)} 条错误（已省略）")
else:
    print("无错误")



🎉 数据整理完成！
====== 指标文件汇总 ======
active_dates_and_times: 281 files
activity: 282 files
activity_details: 282 files
attention: 282 files
bus_factor: 281 files
bus_factor_detail: 281 files
change_request_age: 281 files
change_request_resolution_duration: 280 files
change_request_response_time: 281 files
change_requests: 280 files
change_requests_accepted: 278 files
change_requests_reviews: 276 files
code_change_lines_add: 280 files
code_change_lines_remove: 280 files
code_change_lines_sum: 280 files
contributor_email_suffixes: 280 files
inactive_contributors: 274 files
issue_age: 272 files
issue_comments: 280 files
issue_resolution_duration: 270 files
issue_response_time: 272 files
issues_and_change_request_active: 281 files
issues_closed: 269 files
issues_new: 271 files
new_contributors: 278 files
new_contributors_detail: 278 files
openrank: 282 files
participants: 282 files
stars: 282 files
technical_fork: 281 files

====== 错误文件 ======
无错误


4、将分类指标json文件格式化

In [4]:
import os
import json

processing_root = "data/processing"
output_root = "data/分类/分类指标"

target_metrics = [
    "activity", "attention", "bus_factor", "change_requests",
     "change_requests_accepted","change_requests_reviews",
    "code_change_lines_add", "code_change_lines_remove",
    "code_change_lines_sum", "inactive_contributors",
    "issue_comments", "issues_and_change_request_active",
    "issues_closed", "issues_new", "new_contributors",
    "openrank", "participants", "stars", "technical_fork",
]

os.makedirs(output_root, exist_ok=True)

summary = {}

for metric in target_metrics:
    metric_dir = os.path.join(processing_root, metric)
    if not os.path.isdir(metric_dir):
        print(f"[WARN] 未找到指标目录：{metric_dir}")
        continue

    out_dir = os.path.join(output_root, metric)
    os.makedirs(out_dir, exist_ok=True)

    count = 0

    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue

        src_file = os.path.join(metric_dir, file)
        dst_file = os.path.join(out_dir, file)

        try:
            with open(src_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            with open(dst_file, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)

            count += 1
        except:
            pass

    summary[metric] = count

print("\n🎉 分类整理完成！")
print("====== 整理汇总 ======")
for metric, count in summary.items():
    print(f"{metric}: {count} files processed")



🎉 分类整理完成！
====== 整理汇总 ======
activity: 282 files processed
attention: 282 files processed
bus_factor: 281 files processed
change_requests: 280 files processed
change_requests_accepted: 278 files processed
change_requests_reviews: 276 files processed
code_change_lines_add: 280 files processed
code_change_lines_remove: 280 files processed
code_change_lines_sum: 280 files processed
inactive_contributors: 274 files processed
issue_comments: 280 files processed
issues_and_change_request_active: 281 files processed
issues_closed: 269 files processed
issues_new: 271 files processed
new_contributors: 278 files processed
openrank: 282 files processed
participants: 282 files processed
stars: 282 files processed
technical_fork: 281 files processed


5、将特殊指标json文件格式化

In [5]:
import os
import json

processing_root = "data/processing"
output_root = "data/分类/特殊指标"

# 要处理的特殊指标
special_metrics = [
    "change_request_age",
    "change_request_resolution_duration",
    "change_request_response_time",
    "issue_age",
    "issue_resolution_duration",
    "issue_response_time",
]

# 创建输出根目录
os.makedirs(output_root, exist_ok=True)

summary = {}  # 用于统计每个指标处理数量

for metric in special_metrics:
    metric_dir = os.path.join(processing_root, metric)
    if not os.path.isdir(metric_dir):
        print(f"[WARN] 未找到指标目录：{metric_dir}")
        continue

    # 输出目录：data/分类/特殊指标/{metric}
    out_dir = os.path.join(output_root, metric)
    os.makedirs(out_dir, exist_ok=True)

    count = 0  # 当前指标文件计数

    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue

        src_file = os.path.join(metric_dir, file)
        dst_file = os.path.join(out_dir, file)

        try:
            with open(src_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 格式化保存
            with open(dst_file, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)

            count += 1

        except:
            pass  # 错误直接跳过，避免打断流程

    summary[metric] = count

# 输出最终统计结果
print("\n🎉 特殊指标分类整理完成！")
print("====== 整理汇总 ======")
for metric, count in summary.items():
    print(f"{metric}: {count} files processed")



🎉 特殊指标分类整理完成！
====== 整理汇总 ======
change_request_age: 281 files processed
change_request_resolution_duration: 280 files processed
change_request_response_time: 281 files processed
issue_age: 272 files processed
issue_resolution_duration: 270 files processed
issue_response_time: 272 files processed


6、将文本指标json文件格式化

In [6]:
import os
import json

processing_root = "data/processing"
output_root = "data/分类/文本指标"

# 需要处理的文本类指标
text_metrics = [
    "activity_details",
    "bus_factor",
]

# 创建根目录
os.makedirs(output_root, exist_ok=True)

summary = {}  # 统计处理数量

for metric in text_metrics:
    metric_dir = os.path.join(processing_root, metric)
    if not os.path.isdir(metric_dir):
        print(f"[WARN] 未找到指标目录：{metric_dir}")
        continue

    # 输出目录，例如 data/分类/文本指标/activity_details
    out_dir = os.path.join(output_root, metric)
    os.makedirs(out_dir, exist_ok=True)

    count = 0

    # 遍历所有 company_project.json
    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue

        src_file = os.path.join(metric_dir, file)
        dst_file = os.path.join(out_dir, file)

        try:
            # 读取 JSON
            with open(src_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 格式化输出
            with open(dst_file, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)

            count += 1

        except:
            pass  # 避免异常刷屏

    summary[metric] = count

# 输出统计结果
print("\n🎉 文本指标整理完成！")
print("====== 整理汇总 ======")
for metric, count in summary.items():
    print(f"{metric}: {count} files processed")



🎉 文本指标整理完成！
====== 整理汇总 ======
activity_details: 282 files processed
bus_factor: 281 files processed


7、裁剪分类数据

In [7]:
import os
import json
import re

root = "data/分类/分类指标"   # 正确的分类指标目录
max_year = 2025
max_month = 10

def is_valid_key(key):
    """
    ✔ 只保留 YYYY-MM 且 <= 2025-10
    ❌ 不保留 YYYY
    ❌ 不保留 YYYYQX
    ❌ 不保留 YYYY-MM-raw
    """

    # 只匹配 YYYY-MM
    m = re.fullmatch(r"(\d{4})-(\d{2})", key)
    if m:
        year = int(m.group(1))
        month = int(m.group(2))

        if year < max_year:
            return True
        if year == max_year and month <= max_month:
            return True
        return False

    # 其他全部不要
    return False


# 开始批量处理
for metric in os.listdir(root):
    metric_dir = os.path.join(root, metric)
    if not os.path.isdir(metric_dir):
        continue

    print(f"\n🔍 处理指标：{metric}")

    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue

        filepath = os.path.join(metric_dir, file)

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 构建只保留 YYYY-MM 的新字典
            new_data = {k: v for k, v in data.items() if is_valid_key(k)}

            # 写回格式化后的 JSON
            with open(filepath, "w", encoding="utf-8") as f:
                json.dump(new_data, f, ensure_ascii=False, indent=4)

        except Exception as e:
            print(f"❌ 错误: {filepath} → {e}")

print("\n🎉 分类指标裁剪完成！（只保留 YYYY-MM，已删除年份、季度、raw、超日期数据）")



🔍 处理指标：activity

🔍 处理指标：attention

🔍 处理指标：bus_factor

🔍 处理指标：change_requests

🔍 处理指标：change_requests_accepted

🔍 处理指标：change_requests_reviews

🔍 处理指标：code_change_lines_add

🔍 处理指标：code_change_lines_remove

🔍 处理指标：code_change_lines_sum

🔍 处理指标：inactive_contributors

🔍 处理指标：issues_and_change_request_active

🔍 处理指标：issues_closed

🔍 处理指标：issues_new

🔍 处理指标：issue_comments

🔍 处理指标：new_contributors

🔍 处理指标：openrank

🔍 处理指标：participants

🔍 处理指标：stars

🔍 处理指标：technical_fork

🎉 分类指标裁剪完成！（只保留 YYYY-MM，已删除年份、季度、raw、超日期数据）


8、裁剪特殊指标数据

In [8]:
import os
import json
import re

root = "data/分类/特殊指标"
max_year = 2025
max_month = 10

def is_valid_month(k):
    """只保留 YYYY-MM，并且 <= 2025-10"""
    m = re.fullmatch(r"(\d{4})-(\d{2})", k)
    if not m:
        return False
    year = int(m.group(1))
    month = int(m.group(2))
    if year < max_year:
        return True
    if year == max_year and month <= max_month:
        return True
    return False


for metric in os.listdir(root):
    metric_dir = os.path.join(root, metric)
    if not os.path.isdir(metric_dir):
        continue

    print(f"\n🔍 处理特殊指标：{metric}")

    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue

        fp = os.path.join(metric_dir, file)

        try:
            with open(fp, "r", encoding="utf-8") as f:
                data = json.load(f)

            new_data = {}

            for field, value_dict in data.items():
                # 每个 field 是 avg/quantile_0/quantile_1/.../levels
                new_dict = {
                    k: v for k, v in value_dict.items()
                    if is_valid_month(k)
                }
                new_data[field] = new_dict

            with open(fp, "w", encoding="utf-8") as f:
                json.dump(new_data, f, ensure_ascii=False, indent=4)

        except Exception as e:
            print(f"❌ 错误处理文件: {fp} → {e}")

print("\n🎉 特殊指标裁剪完成！（仅保留 YYYY-MM，且保留所有分位结构）")



🔍 处理特殊指标：change_request_age

🔍 处理特殊指标：change_request_resolution_duration

🔍 处理特殊指标：change_request_response_time

🔍 处理特殊指标：issue_age

🔍 处理特殊指标：issue_resolution_duration

🔍 处理特殊指标：issue_response_time

🎉 特殊指标裁剪完成！（仅保留 YYYY-MM，且保留所有分位结构）


9、将分类指标合成一个长表

In [9]:
import os
import json
import pandas as pd

# 分类指标目录
root = "data/分类/分类指标"

rows = []

# 遍历每个指标目录
for metric in os.listdir(root):
    metric_dir = os.path.join(root, metric)
    if not os.path.isdir(metric_dir):
        continue
    
    # 遍历每个 project 文件
    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue
        
        # 文件名格式： company_project.json
        filename = file.replace(".json", "")
        if "_" not in filename:
            print(f"文件名格式异常：{file}")
            continue
        
        company, project = filename.split("_", 1)
        filepath = os.path.join(metric_dir, file)
        
        # 读取 JSON 数据
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        # 分类指标格式是扁平的 { "YYYY-MM": value }
        for date, value in data.items():
            rows.append({
                "company": company,
                "project": project,
                "metric": metric,
                "date": date,
                "value": value
            })

# 汇总为 DataFrame
df = pd.DataFrame(rows)

# 排序更美观
df = df.sort_values(by=["company", "project", "metric", "date"])

# 输出路径
output_path = "category_metrics_long.xlsx"

# 保存为 Excel
df.to_excel(output_path, index=False)

print("🎉 合并完成！文件已生成：", output_path)
print("总行数：", len(df))


🎉 合并完成！文件已生成： category_metrics_long.xlsx
总行数： 414158


10、将特殊指标合成一个长表

In [10]:
import os
import json
import pandas as pd

# 特殊指标目录
root = "data/分类/特殊指标"

rows = []

# 遍历每个特殊指标子目录
for metric in os.listdir(root):
    metric_dir = os.path.join(root, metric)
    if not os.path.isdir(metric_dir):
        continue
    
    # 遍历每个 company_project.json
    for file in os.listdir(metric_dir):
        if not file.endswith(".json"):
            continue
        
        filename = file.replace(".json", "")
        if "_" not in filename:
            print(f"文件名格式异常：{file}")
            continue
        
        company, project = filename.split("_", 1)
        filepath = os.path.join(metric_dir, file)
        
        # 读取 JSON
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        # data 的结构: { "avg": {date: value}, "quantile_0": {...}, "levels": {...} }
        for field, date_dict in data.items():
            if not isinstance(date_dict, dict):
                continue

            for date, value in date_dict.items():
                
                # levels 是数组，需要转字符串存储
                if isinstance(value, list):
                    value = json.dumps(value, ensure_ascii=False)

                rows.append({
                    "company": company,
                    "project": project,
                    "metric": metric,
                    "field": field,     # avg / quantile_0~4 / levels
                    "date": date,
                    "value": value
                })

# 汇总为 DataFrame
df = pd.DataFrame(rows)

# 排序
df = df.sort_values(by=["company", "project", "metric", "field", "date"])

# 保存为 Excel
output_path = "special_metrics_long.xlsx"
df.to_excel(output_path, index=False)

print("🎉 特殊指标合并完成！文件已生成：", output_path)
print("总行数：", len(df))


🎉 特殊指标合并完成！文件已生成： special_metrics_long.xlsx
总行数： 899178


11、裁剪数据保留2021-01——2025-10

In [11]:
import pandas as pd
import re

# ===== 输入文件 =====
category_file = "category_metrics_long.xlsx"
special_file = "special_metrics_long.xlsx"

# ===== 输出文件 =====
category_out = "category_metrics_long_trimmed.xlsx"
special_out = "special_metrics_long_trimmed.xlsx"

# ===== 时间范围 =====
start = ("2021", "01")
end = ("2025", "10")


def valid_date(date):
    """
    仅保留 YYYY-MM 且在范围内
    """
    m = re.fullmatch(r"(\d{4})-(\d{2})", date)
    if not m:
        return False

    year, month = int(m.group(1)), int(m.group(2))

    # 起始
    if (year < int(start[0])) or (year == int(start[0]) and month < int(start[1])):
        return False

    # 结束
    if (year > int(end[0])) or (year == int(end[0]) and month > int(end[1])):
        return False

    return True


def trim_df(df):
    """裁剪 date 列"""
    df = df[df["date"].apply(valid_date)]
    df = df.sort_values(by=["company", "project", "metric", "date"])
    return df


# ===== 处理分类指标 =====
df_cat = pd.read_excel(category_file)
df_cat_trim = trim_df(df_cat)
df_cat_trim.to_excel(category_out, index=False)

print("✔ 分类指标裁剪完成，输出：", category_out)
print("剩余行数：", len(df_cat_trim))


# ===== 处理特殊指标 =====
df_sp = pd.read_excel(special_file)
df_sp_trim = trim_df(df_sp)
df_sp_trim.to_excel(special_out, index=False)

print("✔ 特殊指标裁剪完成，输出：", special_out)
print("剩余行数：", len(df_sp_trim))


✔ 分类指标裁剪完成，输出： category_metrics_long_trimmed.xlsx
剩余行数： 255243
✔ 特殊指标裁剪完成，输出： special_metrics_long_trimmed.xlsx
剩余行数： 557487


12、检查是否有月份缺失

In [ ]:
import pandas as pd

# ===== 输入文件 =====
category_file = "category_metrics_long_trimmed.xlsx"
special_file = "special_metrics_long_trimmed.xlsx"

# ===== 生成完整月份列表 =====
date_range = pd.date_range(start="2021-01-01", end="2025-10-01", freq="MS")
date_list = [d.strftime("%Y-%m") for d in date_range]


# ===== 检查分类指标缺失 =====
def check_category_missing(df):
    results = []

    grouped = df.groupby(["company", "project", "metric"])
    for (company, project, metric), group in grouped:
        existing_dates = set(group["date"].tolist())
        missing = [d for d in date_list if d not in existing_dates]

        results.append({
            "company": company,
            "project": project,
            "metric": metric,
            "missing_months": missing,
            "missing_count": len(missing)
        })

    return pd.DataFrame(results)


# ===== 检查特殊指标缺失 =====
def check_special_missing(df):
    results = []

    grouped = df.groupby(["company", "project", "metric", "field"])
    for (company, project, metric, field), group in grouped:
        existing_dates = set(group["date"].tolist())
        missing = [d for d in date_list if d not in existing_dates]

        results.append({
            "company": company,
            "project": project,
            "metric": metric,
            "field": field,
            "missing_months": missing,
            "missing_count": len(missing)
        })

    return pd.DataFrame(results)


# ===== 读取数据 =====
df_cat = pd.read_excel(category_file)
df_sp = pd.read_excel(special_file)

# ===== 执行检查 =====
cat_missing = check_category_missing(df_cat)
sp_missing = check_special_missing(df_sp)

# ===== 保存详细报告 =====
cat_missing.to_excel("category_missing_report.xlsx", index=False)
sp_missing.to_excel("special_missing_report.xlsx", index=False)

# ===== 极简总结 =====
cat_missing_count = (cat_missing["missing_count"] > 0).sum()
sp_missing_count = (sp_missing["missing_count"] > 0).sum()

print("\n🎯 极简缺失总结（仅数量）")
print(f"分类指标：共有 {cat_missing_count} 条记录存在缺失")
print(f"特殊指标：共有 {sp_missing_count} 条记录存在缺失")

print("\n📌 详细缺失情况见：category_missing_report.xlsx")
print("📌 详细缺失情况见：special_missing_report.xlsx\n")


13、保留14类指标

In [5]:
import pandas as pd

# ===== 输入文件 =====
category_file = "category_metrics_long_trimmed.xlsx"
special_file = "special_metrics_long_trimmed.xlsx"

# ===== 最终需要保留的指标 =====
keep_metrics = [
    "issues_new",
    "issues_closed",
    "issue_comments",
    "issue_response_time",
    "issue_resolution_duration",
    "change_requests",
    "change_requests_reviews",
    "change_requests_accepted",
    "change_request_response_time",
    "change_request_resolution_duration",
    "stars",
    "technical_fork",
    "new_contributors",
    "openrank",
]

# ===== 读取两张表 =====
df_cat = pd.read_excel(category_file)
df_sp = pd.read_excel(special_file)

# ===== 过滤分类指标 =====
df_cat_filtered = df_cat[df_cat["metric"].isin(keep_metrics)]

# ===== 过滤特殊指标 =====
df_sp_filtered = df_sp[df_sp["metric"].isin(keep_metrics)]

# ===== 导出新表 =====
df_cat_filtered.to_excel("category_metrics_filtered.xlsx", index=False)
df_sp_filtered.to_excel("special_metrics_filtered.xlsx", index=False)

print("🎉 已完成过滤，仅保留 13 类指标：")
print("✔ category_metrics_filtered.xlsx")
print("✔ special_metrics_filtered.xlsx")


🎉 已完成过滤，仅保留 13 类指标：
✔ category_metrics_filtered.xlsx
✔ special_metrics_filtered.xlsx


14、缺失值处理

In [ ]:
import pandas as pd

# ==== 文件名 ====
cat_file = "category_metrics_filtered.xlsx"
sp_file = "special_metrics_filtered.xlsx"

# ==== 读取 ====
df_cat = pd.read_excel(cat_file)
df_sp = pd.read_excel(sp_file)

# ==== 月份范围 ====
date_range = pd.date_range(start="2021-01-01", end="2025-10-01", freq="MS")
date_list = [d.strftime("%Y-%m") for d in date_range]


# ============================================
# ========== 缺失值处理函数 ====================
# ============================================

def fill_missing_category(df):
    filled_data = []

    grouped = df.groupby(["company", "project", "metric"])
    for (company, project, metric), g in grouped:

        # 构造完整时间轴
        full_idx = pd.DataFrame({"date": date_list})
        g2 = full_idx.merge(g, on="date", how="left")

        # 前向填充 + 后向填充
        g2["value"] = g2["value"].ffill().bfill()

        g2["company"] = company
        g2["project"] = project
        g2["metric"] = metric

        filled_data.append(g2)

    return pd.concat(filled_data, ignore_index=True)


def fill_missing_special(df):
    filled_data = []

    grouped = df.groupby(["company", "project", "metric", "field"])
    for (company, project, metric, field), g in grouped:

        full_idx = pd.DataFrame({"date": date_list})
        g2 = full_idx.merge(g, on="date", how="left")

        g2["value"] = g2["value"].ffill().bfill()

        g2["company"] = company
        g2["project"] = project
        g2["metric"] = metric
        g2["field"] = field

        filled_data.append(g2)

    return pd.concat(filled_data, ignore_index=True)


# ============================================
# ========== 执行缺失处理 ======================
# ============================================

df_cat_filled = fill_missing_category(df_cat)
df_sp_filled = fill_missing_special(df_sp)

# 导出
df_cat_filled.to_excel("category_metrics_filled.xlsx", index=False)
df_sp_filled.to_excel("special_metrics_filled.xlsx", index=False)

print("🎉 缺失值填充完成！")
print("输出：category_metrics_final.xlsx")
print("输出：special_metrics_final.xlsx")


15、将level和quantile剔除

In [7]:
import pandas as pd

# 输入文件
sp_file = "special_metrics_filled.xlsx"

# 读取
df_sp = pd.read_excel(sp_file)

# --- 主表（仅 avg） ---
df_avg = df_sp[df_sp["field"] == "avg"].copy()
df_avg["metric"] = df_avg["metric"] + "_avg"
df_avg = df_avg[["company", "project", "metric", "date", "value"]]
df_avg.to_excel("special_metrics_avg.xlsx", index=False)

# --- 分表（levels + quantiles） ---
df_lq = df_sp[df_sp["field"] != "avg"].copy()
df_lq.to_excel("special_metrics_levels_quantiles.xlsx", index=False)

print("🎉 特殊指标成功拆分！")
print("输出:")
print(" - special_metrics_avg.xlsx （参与计算）")
print(" - special_metrics_levels_quantiles.xlsx （分位数用于分析）")


🎉 特殊指标成功拆分！
输出:
 - special_metrics_avg.xlsx （参与计算）
 - special_metrics_levels_quantiles.xlsx （分位数用于分析）


16、合并两张表

In [8]:
import pandas as pd

# ===== 输入文件 =====
category_file = "category_metrics_filled.xlsx"
special_avg_file = "special_metrics_avg.xlsx"

# ===== 输出文件 =====
output_master = "master_long_table.xlsx"

# ===== 读取两张表 =====
df_cat = pd.read_excel(category_file)
df_sp = pd.read_excel(special_avg_file)

# ===== 统一列结构 =====
df_cat = df_cat[["company", "project", "metric", "date", "value"]]
df_sp = df_sp[["company", "project", "metric", "date", "value"]]

# ===== 合并两张表 =====
master = pd.concat([df_cat, df_sp], ignore_index=True)

# ===== 排序（增强可读性）=====
master = master.sort_values(["company", "project", "metric", "date"])

# ===== 输出 =====
master.to_excel(output_master, index=False)

print("🎉 master 长表合并完成！")
print("输出文件：", output_master)


🎉 master 长表合并完成！
输出文件： master_long_table.xlsx


17、计算各聚合指标

In [14]:
import pandas as pd
import numpy as np

# ===== 输入 / 输出 =====
MASTER_FILE = "master_long_table.xlsx"
OUTPUT_FILE = "project_scores.xlsx"

# ===== 读取长表 =====
df = pd.read_excel(MASTER_FILE)

df["value"] = pd.to_numeric(df["value"], errors="coerce")

# ===== 转为宽表 =====
df_wide = df.pivot_table(
    index=["company", "project", "date"],
    columns="metric",
    values="value",
    aggfunc="first"
).reset_index()

df_wide.columns.name = None

# ===== 工具函数 =====
def col(name):
    if name in df_wide.columns:
        return df_wide[name].fillna(0)
    else:
        return pd.Series(0, index=df_wide.index, dtype=float)

def min_max_norm(series):
    s = series.astype(float)
    mn = s.min()
    mx = s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0, index=s.index, dtype=float)
    return (s - mn) / (mx - mn)

def reverse_score(series):
    return 1 - min_max_norm(series)

# ====================================================
# ① 项目活跃度
# ====================================================
df_wide["project_activity_index"] = (
    0.4 * col("issues_new") +
    0.4 * col("change_requests") +
    0.1 * col("issue_comments") +
    0.1 * col("change_requests_reviews")
)

# ====================================================
# ② 开发者活跃度
# ====================================================
df_wide["developer_activity_index"] = (
    0.5 * col("new_contributors") +
    0.3 * (col("issue_comments") + col("change_requests_reviews")) +
    0.2 * (col("issues_new") + col("change_requests"))
)

# ====================================================
# ③ 关注度
# ====================================================
df_wide["attention_index"] = (
    0.4 * col("stars") +
    0.6 * col("technical_fork")
)

# ====================================================
# ④ PR 处理效率（最终：log 压缩 + 二次归一化 + 维度*100 + 功效系数）
# ====================================================

def std_norm(series):
    s = series.astype(float)
    std = (s - s.mean()) / (s.std() if s.std() != 0 else 1)
    return min_max_norm(std)

def log_compress(series):
    return np.log1p(series)  # log(x + 1)

cr_cnt = col("change_requests").replace(0, np.nan)

# --- 4.1 响应效率（log 压缩 → 反向评分 → 二次归一 → ×100）---
resp_time = col("change_request_response_time_avg")
resp_log = log_compress(resp_time)
resp_raw = reverse_score(resp_log)
df_wide["pr_response_score"] = std_norm(resp_raw) * 100

# --- 4.2 处理效率 ---
res_time = col("change_request_resolution_duration_avg")
res_log = log_compress(res_time)
res_raw = reverse_score(res_log)
df_wide["pr_resolution_score"] = std_norm(res_raw) * 100

# --- 4.3 审阅充分度（reviews / PR 数）---
review_intensity = (col("change_requests_reviews") / cr_cnt).replace([np.inf,-np.inf],np.nan).fillna(0)
df_wide["pr_review_score"] = std_norm(review_intensity) * 100

# --- 4.4 接受率 ---
accept_rate = (col("change_requests_accepted") / cr_cnt).replace([np.inf,-np.inf],np.nan).fillna(0)
df_wide["pr_accept_score"] = std_norm(accept_rate) * 100

# --- 4.5 PREI_raw: 四维度组合 (仍然 0~1 范围内) ---
PREI_raw = (
    0.35 * (df_wide["pr_response_score"] / 100) +
    0.35 * (df_wide["pr_resolution_score"] / 100) +
    0.15 * (df_wide["pr_review_score"] / 100) +
    0.15 * (df_wide["pr_accept_score"] / 100)
)

# --- 4.6 PREI_final：功效系数 60~100 ---
PREI_norm = min_max_norm(PREI_raw)
df_wide["pr_efficiency_index"] = 60 + 40 * PREI_norm


# ===== 导出 =====
df_wide.to_excel(OUTPUT_FILE, index=False)
print("🎉 Github 指数（归一化）计算完成！")
print("输出文件：", OUTPUT_FILE)


🎉 Github 指数（归一化）计算完成！
输出文件： project_scores.xlsx


18、Github指数生成

In [13]:
import pandas as pd
import numpy as np

MASTER_FILE = "master_long_table.xlsx"
OUTPUT_PROJECT = "project_github_scores.xlsx"

# ===== 读取长表（公司、项目、日期、metric、value） =====
df = pd.read_excel(MASTER_FILE)

df["value"] = pd.to_numeric(df["value"], errors="coerce")

# ===== 仅保留2021-01 ~ 2025-10 ===========
df = df[(df["date"] >= "2021-01") & (df["date"] <= "2025-10")]

# ==============================================================
#   小工具：min-max 归一化（项目级）
# ==============================================================

def min_max_norm(series):
    s = series.astype(float)
    mn = s.min()
    mx = s.max()
    if mx == mn:
        return pd.Series(0, index=s.index, dtype=float)
    return (s - mn) / (mx - mn)


# ==============================================================
#   Step 1： 转为宽表 (company, project, date → metrics columns)
# ==============================================================

df_wide = df.pivot_table(
    index=["company", "project", "date"],
    columns="metric",
    values="value",
    aggfunc="first"
).reset_index()

df_wide.columns.name = None


# ==============================================================
#   Step 2：生成项目级聚合表（总和/平均）
# ==============================================================

g = df_wide.groupby(["company", "project"])

# ---- 影响力四个指标（取总和）----
stars_sum = g["stars"].sum()
fork_sum = g["technical_fork"].sum()
issue_new_sum = g["issues_new"].sum()
change_requests_sum = g["change_requests"].sum()

# ---- 社区反应（部分总和 + 部分平均）----
issues_closed_sum = g["issues_closed"].sum()
pr_accept_sum = g["change_requests_accepted"].sum()
issue_res_avg = g["issue_resolution_duration_avg"].mean()
pr_res_avg = g["change_request_resolution_duration_avg"].mean()

# ---- 开发者活跃度三个指标（总和）----
issue_new_sum = g["issues_new"].sum()
change_requests_sum = g["change_requests"].sum()
new_contributors_sum = g["new_contributors"].sum()

# ---- 趋势维度（平均增长率）----
def growth(series):
    s = series.fillna(0)
    gr = s.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0)
    return gr.mean()

trend_issue = g["issues_new"].apply(growth)
trend_pr = g["change_requests"].apply(growth)
trend_dev = g["new_contributors"].apply(growth)

trend_avg = 0.4*trend_issue + 0.4*trend_pr + 0.2*trend_dev


# ==============================================================
# Step 3：计算四个维度的项目级指数
# ==============================================================

project_df = pd.DataFrame({
    "company": stars_sum.index.get_level_values(0),
    "project": stars_sum.index.get_level_values(1),

    # 影响力指标原始值
    "influence_raw": (
        0.25*stars_sum +
        0.25*fork_sum +
        0.30*issue_new_sum +
        0.20*change_requests_sum
    ),

    # 社区反应
    "reaction_raw": (
        0.5*issues_closed_sum +
        0.2*pr_accept_sum +
        0.2*(1 - min_max_norm(issue_res_avg)) +
        0.1*(1 - min_max_norm(pr_res_avg))
    ),

    # 开发者活跃度
    "developer_raw": (
        0.4 * issue_new_sum +
        0.3 * change_requests_sum +
        0.3 * new_contributors_sum
    ),

    # 趋势（已是浮点数）
    "trend_raw": trend_avg.values
})

# ==============================================================
# Step 4：四维度归一化 → 平方根平滑 → 加权 → 功效系数
# ==============================================================

# 归一化
inf_n = min_max_norm(project_df["influence_raw"])
react_n = min_max_norm(project_df["reaction_raw"])
dev_n = min_max_norm(project_df["developer_raw"])
trend_n = min_max_norm(project_df["trend_raw"])

# 平方根平滑（√x）
inf_s = np.sqrt(inf_n)
react_s = np.sqrt(react_n)
dev_s = np.sqrt(dev_n)
trend_s = np.sqrt(trend_n)

# 加权求综合得分（范围仍 0~1）
combined = (
    0.3*inf_s +
    0.2*react_s +
    0.2*dev_s +
    0.3*trend_s
)

# 功效系数
project_df["github_index"] = 60 + 40*combined

# 四维度平滑值 ×100，用于可视化展示
project_df["influence_index"] = inf_s * 100
project_df["reaction_index"] = react_s * 100
project_df["developer_index"] = dev_s * 100
project_df["trend_index"] = trend_s * 100

# ==============================================================
#   Step 5：导出项目级最终表
# ==============================================================

project_df.to_excel(OUTPUT_PROJECT, index=False)
print("🎉 项目级 GitHub 指数计算完成！")
print("输出文件：", OUTPUT_PROJECT)


🎉 项目级 GitHub 指数计算完成！
输出文件： project_github_scores.xlsx


19、合成json

In [22]:
import pandas as pd
import json

PROJECT_SCORES = "project_scores.xlsx"
GITHUB_SCORES = "project_github_scores.xlsx"
OUTPUT_FILE = "final_project_data.json"

# ---------------------------------------------------------
# 1) 读取数据（宽表格式）
# ---------------------------------------------------------
df_proj = pd.read_excel(PROJECT_SCORES)
df_git = pd.read_excel(GITHUB_SCORES)

num_cols = df_proj.columns.difference(["company", "project", "date"])
df_proj[num_cols] = df_proj[num_cols].apply(pd.to_numeric, errors="coerce")

# ---------------------------------------------------------
# 2) 全局平均
# ---------------------------------------------------------
openrank_avg = round(df_proj["openrank"].mean(), 2)
github_avg = round(df_git["github_index"].mean(), 2)

# ---------------------------------------------------------
# 3) 月度指标
# ---------------------------------------------------------
monthly_metrics = [
    "openrank",
    "project_activity_index",
    "developer_activity_index",
    "attention_index",
    "pr_efficiency_index",
    "pr_response_score",
    "pr_resolution_score",
    "pr_review_score",
    "pr_accept_score"
]

# ---------------------------------------------------------
# 4) 构建 JSON（字段顺序固定）
# ---------------------------------------------------------
result = []
projects = df_proj[["company", "project"]].drop_duplicates().reset_index(drop=True)

project_id = 1

for _, row in projects.iterrows():
    c = row["company"]
    p = row["project"]

    # ---- 固定字段顺序，全部预函数 ----
    entry = {
        "project_id": project_id,
        "company": c,
        "project": p,
        "openrank_avg": openrank_avg,
        "github_avg": github_avg,

        "github_index": None,
        "influence_index": None,
        "reaction_index": None,
        "developer_index": None,
        "trend_index": None,

        "pr_response_score": {},
        "pr_resolution_score": {},
        "pr_review_score": {},
        "pr_accept_score": {},

        # 🔥 一定要把 developer_activity_index 等放在这里！
        "openrank": {},
        "project_activity_index": {},
        "developer_activity_index": {},
        "attention_index": {},
        "pr_efficiency_index": {},
    }

    # ---- 填充月度数据 ----
    df_sub = df_proj[(df_proj.company == c) & (df_proj.project == p)]

    for metric in monthly_metrics:
        series = df_sub[["date", metric]].dropna()
        entry[metric] = {
            d: round(float(v), 2) for d, v in zip(series["date"], series[metric])
        }

    # ---- GitHub 四维度 ----
    git_row = df_git[(df_git.company == c) & (df_git.project == p)]
    if not git_row.empty:
        gr = git_row.iloc[0]
        entry["github_index"] = round(float(gr["github_index"]), 2)
        entry["influence_index"] = round(float(gr["influence_index"]), 2)
        entry["reaction_index"]  = round(float(gr["reaction_index"]), 2)
        entry["developer_index"] = round(float(gr["developer_index"]), 2)
        entry["trend_index"]     = round(float(gr["trend_index"]), 2)

    result.append(entry)
    project_id += 1

# ---------------------------------------------------------
# 5) 保存 JSON
# ---------------------------------------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

print("🎉 完整字段 JSON 已生成，不会再丢字段！")
print("输出文件：", OUTPUT_FILE)


🎉 完整字段 JSON 已生成，不会再丢字段！
输出文件： final_project_data.json


In [3]:
import json
import pymysql

# 配置你的 MySQL
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "123456"
MYSQL_DB = "pro_web_database"

# 输入数据文件
INPUT_FILE = "final_project_data.json"


def convert_to_old_structure(entry):
    """将数据转换成旧系统结构"""
    return {
        "project_id": entry.get("project_id"),
        "company_name": entry.get("company"),
        "project_name": entry.get("project"),

        "influence": entry.get("influence_index"),
        "response": entry.get("reaction_index"),
        "activity": entry.get("developer_index"),
        "trend": entry.get("trend_index"),
        "github": entry.get("github_index"),

        "project_activity": json.dumps(entry.get("project_activity_index", {}), ensure_ascii=False),
        "project_attention": json.dumps(entry.get("attention_index", {}), ensure_ascii=False),
        "developer_activity": json.dumps(entry.get("developer_activity_index", {}), ensure_ascii=False),
        "openrank": json.dumps(entry.get("openrank", {}), ensure_ascii=False),
        "openrank_avg": entry.get("openrank_avg"),

        "prei": json.dumps(entry.get("pr_efficiency_index", {}), ensure_ascii=False),
        "prei_review_index": json.dumps(entry.get("pr_review_score", {}), ensure_ascii=False),
        "prei_response_index": json.dumps(entry.get("pr_response_score", {}), ensure_ascii=False),
        "prei_accept_index": json.dumps(entry.get("pr_accept_score", {}), ensure_ascii=False),
        "prei_merge_index": json.dumps(entry.get("pr_resolution_score", {}), ensure_ascii=False),
    }


def insert_data(connection, data):
    """批量插入到 MySQL"""
    sql = """
    INSERT INTO github (
        project_id, company_name, project_name,
        influence, response, activity, trend, github,
        project_activity, project_attention, developer_activity, openrank, openrank_avg,
        prei, prei_review_index, prei_response_index, prei_accept_index, prei_merge_index
    ) VALUES (
        %(project_id)s, %(company_name)s, %(project_name)s,
        %(influence)s, %(response)s, %(activity)s, %(trend)s, %(github)s,
        %(project_activity)s, %(project_attention)s, %(developer_activity)s,
        %(openrank)s, %(openrank_avg)s,
        %(prei)s, %(prei_review_index)s, %(prei_response_index)s,
        %(prei_accept_index)s, %(prei_merge_index)s
    )
    """

    with connection.cursor() as cursor:
        cursor.executemany(sql, data)
        connection.commit()


def main():
    # 读取 json
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    # 转换数据结构
    converted_data = [convert_to_old_structure(entry) for entry in raw_data]

    # 连接 MySQL
    conn = pymysql.connect(
        host="49.235.74.98",
        user="remote",
        password="Zhjh0704.",
        database="opendigger",
        charset="utf8mb4"
    )

    insert_data(conn, converted_data)
    conn.close()

    print("数据转换 + 导入完成！")


if __name__ == "__main__":
    main()


数据转换 + 导入完成！


In [3]:
import pandas as pd
import numpy as np
import json

PROJECT_SCORES = "project_scores.xlsx"
GITHUB_SCORES = "project_github_scores.xlsx"

# ---- 读取数据 ----
df_proj = pd.read_excel(PROJECT_SCORES)
df_git  = pd.read_excel(GITHUB_SCORES)

# 小工具：保留两位小数
def r2(x):
    return round(float(x), 2)

# ------------------------------------------
# 1) GitHub 四维度 baseline（raw）
# ------------------------------------------
github_raw_baseline = {
    "influence_raw": {
        "min": r2(df_git["influence_raw"].min()),
        "max": r2(df_git["influence_raw"].max())
    },
    "reaction_raw": {
        "min": r2(df_git["reaction_raw"].min()),
        "max": r2(df_git["reaction_raw"].max())
    },
    "developer_raw": {
        "min": r2(df_git["developer_raw"].min()),
        "max": r2(df_git["developer_raw"].max())
    },
    "trend_raw": {
        "min": r2(df_git["trend_raw"].min()),
        "max": r2(df_git["trend_raw"].max())
    }
}

# ------------------------------------------
# 2) PREI 四维度 baseline（minmax2 输入：score/100）
# ------------------------------------------
prei_baseline = {}
cols = ["pr_response_score", "pr_resolution_score", "pr_review_score", "pr_accept_score"]

for col in cols:
    raw_values = df_proj[col] / 100.0
    key = col.replace("pr_", "").replace("_score", "")
    prei_baseline[key] = {
        "min": r2(raw_values.min()),
        "max": r2(raw_values.max())
    }

# ------------------------------------------
# 3) 合并并保存 JSON
# ------------------------------------------
baseline_all = {
    "github_raw_baseline": github_raw_baseline,
    "prei_raw_baseline": prei_baseline
}

with open("baseline.json", "w", encoding="utf-8") as f:
    json.dump(baseline_all, f, ensure_ascii=False, indent=4)

print("🎉 baseline.json 生成完成！")


🎉 baseline.json 生成完成！


In [4]:
import pymysql
import json

# ---- 读取 baseline ----
with open("baseline.json", "r", encoding="utf-8") as f:
    baseline_data = f.read()   # 直接存 JSON 字符串

# ---- 连接数据库 ----
conn = pymysql.connect(
    host="49.235.74.98",
    user="remote",
    password="Zhjh0704.",
    database="opendigger",
    charset="utf8mb4"
)
cursor = conn.cursor()

# ---- 插入 baseline ----
sql = "INSERT INTO baseline_config (baseline) VALUES (%s)"
cursor.execute(sql, (baseline_data,))
conn.commit()

cursor.close()
conn.close()

print("🎉 baseline.json 已成功写入数据库 baseline_config 表！")


🎉 baseline.json 已成功写入数据库 baseline_config 表！
